In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("samuelotiattakorah/agriculture-crop-yield")

print("Path to dataset files:", path)

100%|██████████| 33.4M/33.4M [00:00<00:00, 65.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/samuelotiattakorah/agriculture-crop-yield/versions/1


In [ ]:
import os
import pandas as pd

In [ ]:
os.listdir(path)

['crop_yield.csv']

In [ ]:
df = pd.read_csv(path + '/crop_yield.csv')

In [ ]:
df

,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Weather_Condition,Days_to_Harvest,Yield_tons_per_hectare
0,West,Sandy,Cotton,897.077239,27.676966,False,True,Cloudy,122,6.555816
1,South,Clay,Rice,992.673282,18.026142,True,True,Rainy,140,8.527341
2,North,Loam,Barley,147.998025,29.794042,False,False,Sunny,106,1.127443
3,North,Sandy,Soybean,986.866331,16.644190,False,True,Rainy,146,6.517573
4,South,Silt,Wheat,730.379174,31.620687,True,True,Cloudy,110,7.248251
...,...,...,...,...,...,...,...,...,...,...
999995,West,Silt,Rice,302.805345,27.987428,False,False,Sunny,76,1.347586
999996,South,Chalky,Barley,932.991383,39.661039,True,False,Rainy,93,7.311594
999997,North,Peaty,Cotton,867.362046,24.370042,True,False,Cloudy,108,5.763182
999998,West,Silt,Wheat,492.812857,33.045505,False,False,Sunny,102,2.070159


In [ ]:
df = df.drop(columns=['Fertilizer_Used','Weather_Condition'])

In [ ]:
cat_col = df.select_dtypes(include=['object'])

In [ ]:
cat_col

,Region,Soil_Type,Crop
0,West,Sandy,Cotton
1,South,Clay,Rice
2,North,Loam,Barley
3,North,Sandy,Soybean
4,South,Silt,Wheat
...,...,...,...
999995,West,Silt,Rice
999996,South,Chalky,Barley
999997,North,Peaty,Cotton
999998,West,Silt,Wheat


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

In [ ]:
encoder = LabelEncoder()

In [ ]:
for col in cat_col:
  df[col] = encoder.fit_transform(df[col])

In [ ]:
df

,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Irrigation_Used,Days_to_Harvest,Yield_tons_per_hectare
0,3,4,1,897.077239,27.676966,True,122,6.555816
1,2,1,3,992.673282,18.026142,True,140,8.527341
2,1,2,0,147.998025,29.794042,False,106,1.127443
3,1,4,4,986.866331,16.644190,True,146,6.517573
4,2,5,5,730.379174,31.620687,True,110,7.248251
...,...,...,...,...,...,...,...,...
999995,3,5,3,302.805345,27.987428,False,76,1.347586
999996,2,0,0,932.991383,39.661039,False,93,7.311594
999997,1,3,1,867.362046,24.370042,False,108,5.763182
999998,3,5,5,492.812857,33.045505,False,102,2.070159


In [ ]:
X = df.drop('Yield_tons_per_hectare',axis=1)
y = df['Yield_tons_per_hectare']


In [ ]:
X


,Region,Soil_Type,Crop,Rainfall_mm,Temperature_Celsius,Irrigation_Used,Days_to_Harvest
0,3,4,1,897.077239,27.676966,True,122
1,2,1,3,992.673282,18.026142,True,140
2,1,2,0,147.998025,29.794042,False,106
3,1,4,4,986.866331,16.644190,True,146
4,2,5,5,730.379174,31.620687,True,110
...,...,...,...,...,...,...,...
999995,3,5,3,302.805345,27.987428,False,76
999996,2,0,0,932.991383,39.661039,False,93
999997,1,3,1,867.362046,24.370042,False,108
999998,3,5,5,492.812857,33.045505,False,102


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = XGBRegressor()

In [ ]:
model.fit(X_train,y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print(r2_score(y_test,y_pred))

0.7154600806028917
